In [37]:
import json
import os
atomic_fact_data = []
atomic_facts_path = os.path.join(os.getcwd(), "data_for_git", "atomic_facts.jsonl")
with open(atomic_facts_path, "r") as f:
    for line in f:
        atomic_fact_data.append(json.loads(line))
original_generations = []
original_generations_path = os.path.join(os.getcwd(), "data_for_git", "responses.jsonl")
with open(original_generations_path, "r") as f:
    for line in f:
        original_generations.append(json.loads(line))


atomic_facts_dict = {}
for query in atomic_fact_data:
    atomic_facts_dict[query["id"]] = query["results"]["sentences_and_atomic_facts"]
len(atomic_facts_dict)

sorted_generations = sorted(original_generations, key=lambda x: len(x["responses"][0]["logprobs"]), reverse=True)

def gnmt_length_penalty(length: int, alpha: float = 0.6) -> float:
    return ((5 + length) ** alpha) / ((5 + 1) ** alpha)
# multiply all logprob my len(logprobs) for every sentence due to an error in the data
for generation in sorted_generations:
    for response in generation["responses"]:
        for logprob in response["logprobs"]:
            logprob["logprob"] *= len(logprob["logprobs"])

In [38]:
original_generation = sorted_generations[2]["responses"][0]["logprobs"]



print("penalty", gnmt_length_penalty(200))
# STUPID CUROSR AI STOP SUGGESTING "LOGPROBS" INSTEAD OF "LOGPROB" IT IS NOT A TYPO!!!
sorted_logprobs = sorted(original_generation, key=lambda x: x["logprob"], reverse=True)
import math
for sentence in sorted_logprobs[:1]:
    print(sentence["text"])
    print(sentence["logprob"])
    print(sum(sentence["logprobs"]) / gnmt_length_penalty(len(sentence["logprobs"])))
    print(math.exp(sentence["logprob"]))


# OOPS!

flattened_original_generation = {}
for query in original_generations:
    for generation in query["responses"]:
        flattened_original_generation[generation["id"]] = generation
print(len(flattened_original_generation))
print(len(atomic_fact_data))

penalty 8.320732617514368
He graduated from Theatre Nepean at the University of Western Sydney with a Bachelor of Arts (Performing Arts) in 1987.
-0.16266301251334425
-0.16266301251334428
0.8498775382593063
3415
3413


In [39]:
query_idx: int = 1

generations = original_generations[query_idx]["responses"]

generations_with_atomic_facts = []
individual_atomic_facts_with_logprobs = []
for generation in generations:
    id = generation["id"]
    atomic_facts = atomic_facts_dict.get(id)
    if atomic_facts == None:
        continue
    for sentence, original in zip(atomic_facts, generation["logprobs"]):
        sentence_with_logprob = {
            "logprob": original["logprob"],
            "atomic_facts": sentence[1]
        }
        for individual_atomic_fact in sentence[1]:
            print(individual_atomic_fact)
            individual_atomic_facts_with_logprobs.append({
                "logprob": original["logprob"],
                "atomic_fact": individual_atomic_fact,
                "from_sequence": id
            })
        generations_with_atomic_facts.append(sentence)

The provided documents do not contain biographical information about Kang Ji-hwan.
The provided documents do not contain any biographical information.
The retrieved documents do not contain any information about Kang Ji-hwan.
The context provided is about Kang Daniel.
Kang Daniel appears to be an artist.
Kang Daniel appears to be an entertainer.
There is no information related to Kang Ji-hwan in the context.
The provided documents do not contain any biographical information about Kang Ji-hwan.
The context is about Kang Daniel.
Kang Daniel appears to be a South Korean singer.
Kang Daniel appears to be a South Korean actor.
There is no information about Kang Ji-hwan.
The provided documents do not contain any information about Kang Ji‑hwan.
The context is about Kang Daniel.
Kang Daniel is a South Korean singer.
Kang Daniel is a South Korean actor.
There is no information about a person named Kang Ji-hwan.
There is no information about Kang Ji-hwan in the given documents.
The provided docu

In [40]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-large-mnli")
model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-large-mnli")

Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [41]:
from numpy import argmax
str1 = "Germany is a country"
str2 = "Germany is a country in europe"
print(f"{model.get_memory_footprint() / 1024**3:.2f} GB")
tokenized = tokenizer([str1, str2], [str2, str1], return_tensors="pt")
id2label = model.config.id2label
with torch.no_grad():
    logits = model(**tokenized).logits
all_probs = torch.nn.functional.softmax(logits, dim=1)
print("all_probs", all_probs[0][0])
for probs in all_probs:
    print(argmax(probs))
    for i, prob in enumerate(probs):
        print(f"{id2label[i]}: {float(prob):.5f}")
        

1.51 GB
all_probs tensor(0.0005)
tensor(1)
CONTRADICTION: 0.00046
NEUTRAL: 0.98505
ENTAILMENT: 0.01449
tensor(2)
CONTRADICTION: 0.00258
NEUTRAL: 0.02859
ENTAILMENT: 0.96883


In [42]:
print(len(individual_atomic_facts_with_logprobs))

24


In [43]:
import tqdm
# cluster the atomic facts
entailment_label = 2
def is_bidirectionally_entailed(atomic_fact1, atomic_fact2):
    tokenized = tokenizer([atomic_fact1, atomic_fact2], [atomic_fact2, atomic_fact1], return_tensors="pt")
    with torch.no_grad():
        logits = model(**tokenized).logits
    all_probs = torch.nn.functional.softmax(logits, dim=1)
    one_way_entailment = all_probs[0]
    reverse_one_way_entailment = all_probs[1]
    return argmax(one_way_entailment) == argmax(reverse_one_way_entailment) == entailment_label

clusters = []
# Use position=0, 1, 2 and leave=True
pbar = tqdm.tqdm(individual_atomic_facts_with_logprobs, desc="Processing Facts")

# Initialize counters
cluster_count = 0
comparison_count = 0

for atomic_fact in pbar:
    merged = False
    
    # Update the display with current stats
    pbar.set_postfix({
        "Clusters": cluster_count, 
        "Comps": comparison_count
    })
    
    for cluster in clusters:
        representative = cluster["representative"]
        
        # Increment comparison counter
        comparison_count += 1
        
        entailment = is_bidirectionally_entailed(atomic_fact["atomic_fact"], representative["atomic_fact"])
        
        if entailment:
            
            if representative["logprob"] < atomic_fact["logprob"]:
                cluster["representative"] = atomic_fact
            cluster["atomic_facts"].append(atomic_fact)
            merged = True
            break
    
    if not merged:
        clusters.append({
            "representative": atomic_fact,
            "atomic_facts": [atomic_fact]
        })
        cluster_count += 1

Processing Facts: 100%|██████████| 24/24 [00:41<00:00,  1.73s/it, Clusters=13, Comps=116]


In [44]:
for cluster in clusters:
    print(cluster["representative"]["atomic_fact"])
    for atomic_fact in cluster["atomic_facts"]:
        print(atomic_fact["atomic_fact"])
    print()

The provided documents do not contain biographical information about Kang Ji-hwan.
The provided documents do not contain biographical information about Kang Ji-hwan.
The retrieved documents do not contain any information about Kang Ji-hwan.
The provided documents do not contain any biographical information about Kang Ji-hwan.
There is no information about Kang Ji-hwan in the given documents.
There is no information given about a person named Kang Ji-hwan.

The provided documents do not contain any biographical information.
The provided documents do not contain any biographical information.

The context is about Kang Daniel.
The context provided is about Kang Daniel.
The context is about Kang Daniel.
The context is about Kang Daniel.

Kang Daniel is an artist.
Kang Daniel appears to be an artist.
Kang Daniel is an artist.
Kang Daniel has been active in the music industry.

Kang Daniel appears to be an entertainer.
Kang Daniel appears to be an entertainer.

There is no information about 

In [45]:
import requests

def is_entailed(atomic_fact1, atomic_fact2, session: requests.Session):
    url = "http://localhost:8080/rerank"
    payload = {
        "query": atomic_fact1,
        "documents": [atomic_fact2],
        "model": "microsoft/deberta-large-mnli",
        "return_documents": False 
    }
    response = session.post(url, json=payload)
    if response.json()["results"][0]["relevance_score"] > 0.5:
        payload = {
            "query": atomic_fact2,
            "documents": [atomic_fact1],
            "model": "microsoft/deberta-large-mnli",
            "return_documents": False 
        }
        response = session.post(url, json=payload)
        if response.json()["results"][0]["relevance_score"] > 0.5:
            return True
    return False


def get_entailed_with_lookahead(atomic_fact, cluster_representatives, session: requests.Session):
    url = "http://localhost:8080/rerank"
    
    payload = {
        "query": atomic_fact,
        "documents": cluster_representatives,
        "model": "microsoft/deberta-large-mnli",
        "return_documents": False 
    }
    
    try:
        response = session.post(url, json=payload)
        response.raise_for_status()
        forward_results = response.json()['results']
    except Exception as e:
        print(f"Rerank API Error: {e}")
        return None

    candidates = sorted(
        [res for res in forward_results if res['relevance_score'] > 0.5],
        key=lambda x: x['relevance_score'],
        reverse=True
    )
    
    if not candidates:
        return None  # No matches found
    
    best_idx = None

    for cand in candidates:
        cluster_idx = cand['index']
        
        cluster_text = cluster_representatives[cluster_idx]
        
        backward_payload = {
            "query": cluster_text,        # The Cluster is now the Premise
            "documents": [atomic_fact],   # The Fact is now the Hypothesis
            "model": "microsoft/deberta-large-mnli",
            "return_documents": False
        }
        
        try:
            back_resp = session.post(url, json=backward_payload)
            back_result = back_resp.json()['results'][0] # Only 1 doc, so take index 0
            backward_score = back_result['relevance_score']
            
            if backward_score > 0.5:
                return cluster_idx             
        except Exception as e:
            print(f"Backward pass error for index {cluster_idx}: {e}")
            continue

    return None

def get_clusters(individual_atomic_facts_with_logprobs):
    clusters = []
    for atomic_fact in individual_atomic_facts_with_logprobs:
        merged = False
        for cluster in clusters:
            representative = cluster["representative"]
            entailment = is_bidirectionally_entailed(atomic_fact["atomic_fact"], representative["atomic_fact"])
            
            if entailment:  
                if representative["logprob"] < atomic_fact["logprob"]:
                    cluster["representative"] = atomic_fact
                cluster["atomic_facts"].append({"logprob": atomic_fact["logprob"], "from_sequence": atomic_fact["from_sequence"]})
                merged = True
                break
        
        if not merged:
            clusters.append({
                "representative": atomic_fact,
                "atomic_facts": [{"logprob": atomic_fact["logprob"], "from_sequence": atomic_fact["from_sequence"]}]
            })

In [96]:
import math
from collections import defaultdict

def calculate_cluster_probabilities_soft_laplace(clusters, all_atomic_facts_flat, alpha=1.0):
    """
    Implements Soft Laplace Smoothing (Rule of Succession).
    Blends 'Soft Counts' (logprobs) with a 'Uniform Prior' (alpha).
    """
    
    # 1. Calculate Soft Counts (Probability Mass) per Cluster
    # We still use Max-Pooling to get the best logprob per sequence
    
    # Step A: Get best logprob per sequence
    sequence_max_logprobs = defaultdict(lambda: float('-inf'))
    for fact in all_atomic_facts_flat:
        seq_id = fact["from_sequence"]
        lp = fact["logprob"] 
        if lp > sequence_max_logprobs[seq_id]:
            sequence_max_logprobs[seq_id] = lp
            
    # Step B: Calculate Total Mass (Z)
    # This is the "Soft N" (Effective Sample Size)
    total_soft_mass = sum(math.exp(lp) for lp in sequence_max_logprobs.values())
    print(total_soft_mass)
    # Step C: Count Total Clusters (K)
    num_clusters = len(clusters)

    # 2. Calculate P(C) with Smoothing
    for cluster in clusters:
        
        # Get the Soft Count for this cluster
        seq_to_best_logprob = defaultdict(lambda: float('-inf'))
        for fact in cluster["atomic_facts"]:
            seq_id = fact["from_sequence"]
            lp = fact["logprob"]
            if lp > seq_to_best_logprob[seq_id]:
                seq_to_best_logprob[seq_id] = lp
        
        cluster_soft_count = sum(math.exp(lp) for lp in seq_to_best_logprob.values())
        
        # --- THE FORMULA ---
        # (Mass + alpha) / (Total Mass + alpha * K)
        alpha = total_soft_mass/num_clusters
        numerator = cluster_soft_count + alpha
        denominator = total_soft_mass + (alpha * 2)#num_clusters)
        
        cluster["probability"] = numerator / denominator

    # Sort
    clusters.sort(key=lambda x: x["probability"], reverse=True)
    
    return clusters

In [97]:
alpha = 1
rated = calculate_cluster_probabilities_soft_laplace(clusters, individual_atomic_facts_with_logprobs, alpha)
print(2/(7))

1.600537763824919
0.2857142857142857


In [98]:
for cluster in rated:
    print(cluster["probability"])
    print("RERPRESENTATIVE: ", cluster["representative"]["atomic_fact"])
    for atomic_fact in cluster["atomic_facts"]:
        print(atomic_fact["atomic_fact"], math.exp(atomic_fact["logprob"]))
    print()


0.8748584198679513
RERPRESENTATIVE:  The provided documents do not contain biographical information about Kang Ji-hwan.
The provided documents do not contain biographical information about Kang Ji-hwan. 0.4218639859821725
The retrieved documents do not contain any information about Kang Ji-hwan. 0.14838907932271705
The provided documents do not contain any biographical information about Kang Ji-hwan. 0.3953925661809227
There is no information about Kang Ji-hwan in the given documents. 0.30332359175399065
There is no information given about a person named Kang Ji-hwan. 0.2235785706954182

0.41045102493902547
RERPRESENTATIVE:  The provided documents do not contain any information about Kang Ji‑hwan.
The provided documents do not contain any information about Kang Ji‑hwan. 0.3174460661695534
The provided documents do not contain any information about Kang Ji-hwan. 0.3174460661695534

0.29509954890253814
RERPRESENTATIVE:  The provided documents do not contain any biographical information.


In [ ]:
import math

# --- 1. Probability Percolation (Fixing the Split Vote) ---
def percolate_cluster_probabilities(clusters, nli_model):
    """
    Allows probability mass to flow between clusters based on Directional Entailment.
    If Cluster A implies Cluster B, then B inherits A's probability mass.
    """
    # Create a copy of probabilities to avoid modifying while iterating
    # (or just simple addition if we assume DAG structure, but N2 is safer for small N)
    original_probs = [c["probability"] for c in clusters]
    new_probs = original_probs[:] # Shallow copy
    
    # Compare every cluster against every other cluster
    for i, cluster_a in enumerate(clusters):
        for j, cluster_b in enumerate(clusters):
            if i == j: continue
            
            # We check Directional Entailment: Does A imply B?
            # rep_a -> rep_b
            rep_a = cluster_a["representative"]["atomic_fact"]
            rep_b = cluster_b["representative"]["atomic_fact"]
            
            # You need a function that returns True/False for ONE-WAY entailment
            # implies(premise, hypothesis)
            if nli_model.implies(rep_a, rep_b):
                # If A implies B, then B is a 'superset' or 'generalization' of A.
                # B gets A's mass added to it.
                new_probs[j] += original_probs[i]

    # Assign new probabilities back to clusters
    # We cap at 1.0 just in case of slight calibration errors
    for i, cluster in enumerate(clusters):
        cluster["percolated_probability"] = min(new_probs[i], 1.0)
    
    return clusters

In [ ]:
import math

def calculate_sequence_uncertainty_scores(sequences, clusters):
    """
    Assigns an uncertainty score to each sequence based on the 
    Semantic Probability of the facts it contains.
    
    Lower Score = Lower Uncertainty (More Confident/Consensual)
    """
    
    # 1. Create a Lookup Map: Fact Instance -> Cluster Probability
    # Since we can't rely on text matching (stripped), we assume the 'atomic_facts' 
    # inside 'clusters' are the SAME dictionary objects as in 'sequences'.
    # If they are copies, we need a unique ID per fact. Assuming objects here:
    
    fact_to_cluster_prob = {}
    
    for cluster in clusters:
        # We clamp probability to avoid log(0)
        # If a cluster has 0 probability (shouldn't happen with proper smoothing), set epsilon.
        p_c = max(cluster.get("probability", 0), 1e-10)
        
        for fact in cluster["atomic_facts"]:
            # Map the specific fact object (or ID) to the cluster's probability
            # Using Python's object identity `id()` is the safest way if they are the same objects
            fact_to_cluster_prob[id(fact)] = p_c

    # 2. Score Each Sequence
    for seq in sequences:
        log_prob_sum = 0.0
        fact_count = 0
        
        # Iterate over ALL facts in the sequence (including duplicates)
        for fact in seq["atomic_facts"]:
            
            # Retrieve the probability of the cluster this fact belongs to
            # If for some reason a fact wasn't clustered (edge case), assume low prob
            p_c = fact_to_cluster_prob.get(id(fact), 1e-10)
            
            log_prob_sum += math.log(p_c)
            fact_count += 1
            
        # 3. Calculate Uncertainty (Negative Log Likelihood)
        if fact_count > 0:
            # We normalize by the number of facts to prevent long sequences 
            # from naturally having higher uncertainty just because they say more things.
            uncertainty = - (log_prob_sum / fact_count)
        else:
            # Edge case: Sequence produced no atomic facts. 
            # High uncertainty, or 0 depending on your philosophy. 
            # Usually implies the model refused to answer or babbled nonsense.
            uncertainty = 5.0 # Arbitrary high penalty
            
        seq["semantic_uncertainty_score"] = uncertainty

    return sequences